# 01 · Data overview and quality checks

Before answering any business question, this notebook checks the size, structure and quirks of the data. Each check leads to a rule used in the rest of the analysis.

**Dataset:** Instacart Market Basket Analysis (Kaggle, 2017): anonymised orders from an online grocer, with no calendar dates.

In [1]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

con = duckdb.connect("../data/instacart.duckdb", read_only=True)

def q(sql):
    """Run a SQL query on the DuckDB database and return a pandas DataFrame."""
    return con.sql(sql).df()

q("SHOW TABLES")

,name
0,aisles
1,departments
2,order_products
3,order_products_prior
4,order_products_train
5,orders
6,products


## 1.1 Table sizes

In [2]:
q("""
SELECT 'orders' AS table_name, COUNT(*) AS n_rows FROM orders
UNION ALL
SELECT 'order_products_prior', COUNT(*) FROM order_products_prior
UNION ALL
SELECT 'order_products_train', COUNT(*) FROM order_products_train
UNION ALL
SELECT 'products', COUNT(*) FROM products
UNION ALL
SELECT 'aisles', COUNT(*) FROM aisles
UNION ALL
SELECT 'departments', COUNT(*) FROM departments
""")

,table_name,n_rows
0,orders,3421083
1,order_products_prior,32434489
2,order_products_train,1384617
3,products,49688
4,aisles,134
5,departments,21


Each row of `order_products_*` is one product inside one order. With 32.4M product rows for 3.4M orders, an order contains about 10 products on average (checked in 1.6).

## 1.2 Orders by `eval_set`

The data comes from a prediction competition, so each customer's orders are split into:
- `prior`: all past orders except the last one
- `train` / `test`: the customer's **last** order (products visible for `train`, hidden for `test`)

In [3]:
q("""
SELECT eval_set, COUNT(*) AS n_orders
FROM orders
GROUP BY eval_set
ORDER BY n_orders DESC
""")

,eval_set,n_orders
0,prior,3214874
1,train,131209
2,test,75000


In [4]:
q("""
SELECT o.eval_set,
       COUNT(DISTINCT o.order_id) AS n_orders,
       COUNT(op.product_id)       AS n_product_rows
FROM orders o
LEFT JOIN order_products op ON o.order_id = op.order_id
GROUP BY o.eval_set
ORDER BY o.eval_set
""")

,eval_set,n_orders,n_product_rows
0,prior,3214874,32434489
1,test,75000,0
2,train,131209,1384617


**Rule:** the 75,000 `test` orders have no products. They are used for timing analysis (day and hour) but not for basket analysis.

## 1.3 Customers and orders per customer

In [5]:
q("""
SELECT COUNT(DISTINCT user_id) AS n_unique_users
FROM orders
""")

,n_unique_users
0,206209


In [6]:
q("""
SELECT MIN(n_orders)    AS min_orders,
       MAX(n_orders)    AS max_orders,
       AVG(n_orders)    AS avg_orders,
       MEDIAN(n_orders) AS median_orders
FROM (
    SELECT user_id, COUNT(*) AS n_orders
    FROM orders
    GROUP BY user_id
) AS orders_per_user
""")

,min_orders,max_orders,avg_orders,median_orders
0,4,100,16.590367,10.0


- 206,209 customers, with **between 4 and 100 orders** each. Customers who stopped after 1–3 orders are not in the data, so loyalty metrics will be optimistic (**selection bias**), and the heaviest customers are capped at 100 orders.
- Orders per customer are right-skewed (mean 16.6 vs median 10): the **median** describes the typical customer better.

## 1.4 Missing values in `days_since_prior_order`

In [7]:
q("""
SELECT COUNT(*) AS n_null_days
FROM orders
WHERE days_since_prior_order IS NULL
""")

,n_null_days
0,206209


In [8]:
q("""
SELECT order_number, COUNT(*) AS n_orders
FROM orders
WHERE days_since_prior_order IS NULL
GROUP BY order_number
ORDER BY order_number
""")

,order_number,n_orders
0,1,206209


All 206,209 NULLs are on `order_number = 1`, exactly one per customer: a first order has no previous order. These are **structural, not missing values**: they are excluded from interval analysis and never filled with 0 (which would mean "reordered the same day").

## 1.5 Distribution of `days_since_prior_order`

In [9]:
gaps = q("""
SELECT days_since_prior_order, COUNT(*) AS n_orders
FROM orders
WHERE days_since_prior_order IS NOT NULL
GROUP BY days_since_prior_order
ORDER BY days_since_prior_order
""")
gaps

,days_since_prior_order,n_orders
0,0.0,67755
1,1.0,145247
2,2.0,193206
3,3.0,217005
4,4.0,221696
5,5.0,214503
6,6.0,240013
7,7.0,320608
8,8.0,181717
9,9.0,118188


In [10]:
gaps["pct_orders"] = (gaps["n_orders"] / gaps["n_orders"].sum() * 100).round(1)
within_week = gaps.loc[gaps["days_since_prior_order"] <= 7, "pct_orders"].sum()
capped = gaps.loc[gaps["days_since_prior_order"] == 30, "pct_orders"].sum()
print(f"Orders within 7 days of the previous one: {within_week:.1f}%")
print(f"Orders at the 30-day cap: {capped:.1f}%")

Orders within 7 days of the previous one: 50.5%
Orders at the 30-day cap: 11.5%


- Clear **weekly rhythm**: peaks at 7 days (320,608 orders), 14 and 21 days. About half of all repeat orders come within 7 days of the previous one.
- The jump from ~19,000 orders at 29 days to 369,323 at 30 days is not natural behaviour: the column is **capped at 30**, so 30 means "30 days or more" (11.5% of orders), not a monthly habit.

## 1.6 Basket size

In [11]:
q("""
SELECT MIN(n_products)    AS min_products,
       MAX(n_products)    AS max_products,
       AVG(n_products)    AS avg_products,
       MEDIAN(n_products) AS median_products
FROM (
    SELECT order_id, COUNT(*) AS n_products
    FROM order_products_prior
    GROUP BY order_id
) AS products_per_order
""")

,min_products,max_products,avg_products,median_products
0,1,145,10.088883,8.0


The typical basket has **8 products** (mean 10.1, max 145): a few large stock-up orders raise the mean. Basket size matters for delivery economics, since the cost of a drop is spread over the items in it.

## 1.7 Column types

In [12]:
q("DESCRIBE orders")

,column_name,column_type,null,key,default,extra
0,order_id,BIGINT,YES,None,None,None
1,user_id,BIGINT,YES,None,None,None
2,eval_set,VARCHAR,YES,None,None,None
3,order_number,BIGINT,YES,None,None,None
4,order_dow,BIGINT,YES,None,None,None
5,order_hour_of_day,VARCHAR,YES,None,None,None
6,days_since_prior_order,DOUBLE,YES,None,None,None


`order_hour_of_day` is stored as text (`"08"`), so it is cast to `INTEGER` before numeric comparisons.

## Data rules used in the next notebooks

| # | Finding | Rule |
|---|---|---|
| 1 | Test orders have no products | Use them only for timing analysis |
| 2 | Customers have 4–100 orders | Loyalty metrics are optimistic: state it |
| 3 | NULL gaps only on first orders | Exclude from interval analysis, never fill with 0 |
| 4 | Gaps capped at 30 days | Read 30 as "30 or more" |
| 5 | First orders have `reordered = 0` by definition | Exclude first orders from reorder rates |
| 6 | Hour stored as text | Cast to integer |